# Regressão Linear Simples e Múltipla em Python
**Curso:** Ciência de Dados — Fatec 2026

---
## 1. Conceitos Fundamentais

A **Regressão Linear** permite gerar um modelo matemático por meio de uma reta (regressão simples) ou hiperplano (regressão múltipla) que explica a relação linear entre variáveis.

* **Variável Independente ($X$ — explicativa ou preditora):** Representa o fator ou característica que observamos ou controlamos para explicar ou prever outra variável. Fica do lado direito da fórmula do modelo.
* **Variável Dependente ($Y$ — resposta ou alvo):** É a variável numérica contínua que queremos prever ou explicar. Fica do lado esquerdo da fórmula do modelo.

### A Função `smf.ols()` (Ordinary Least Squares)
Em Python, a biblioteca **`statsmodels`** oferece a função `smf.ols()` (`statsmodels.formula.api`), que ajusta modelos lineares usando o **Método dos Mínimos Quadrados Ordinários** com a mesma sintaxe de fórmula utilizada na estatística clássica (`y ~ x1 + x2`), retornando coeficientes, resíduos, $R^2$, testes $t$ e $p$-valores para inferência estatística.

In [ ]:
# Importação das bibliotecas necessárias
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf

# Configurações visuais padrão
plt.rcParams['figure.figsize'] = (8, 5)
sns.set_theme(style='whitegrid')

---
## 2. Projeto Cafeteria

Queremos entender como o **preço do café**, o **preço do leite** (produto complementar) e a realização de **promoções** influenciam a quantidade vendida de café (`Vendas_Cafe`).

> **Nota:** Caso você ainda não tenha o arquivo `dados_cafeteria.csv` carregado no ambiente do Google Colab, a célula abaixo cria automaticamente uma base simulada fiel aos parâmetros da aula para que todo o notebook possa ser executado sem erros.

In [ ]:
import os

# Carregar dados_cafeteria.csv ou gerar base equivalente caso não esteja no ambiente
if os.path.exists('dados_cafeteria.csv'):
    dados = pd.read_csv('dados_cafeteria.csv')
else:
    np.random.seed(42)
    n = 30
    preco_cafe = np.round(np.random.uniform(3.75, 4.80, n), 2)
    preco_leite = np.round(np.random.uniform(4.00, 4.80, n), 2)
    promocao = np.random.choice(['Sim', 'Nao'], size=n, p=[0.4, 0.6])
    erro = np.random.normal(0, 2.5, n)
    vendas_cafe = np.round(151.31 - 18.72 * preco_cafe - 8.78 * preco_leite + np.where(promocao == 'Sim', 4.5, 0) + erro).astype(int)
    dados = pd.DataFrame({
        'Preco_Cafe': preco_cafe,
        'Preco_Leite': preco_leite,
        'Promocao': promocao,
        'Vendas_Cafe': vendas_cafe
    })

# Visualizar os primeiros registros
dados.head()

### 2.1 Estatística Descritiva e Análise Exploratória

In [ ]:
# Resumo estatístico (média, desvio padrão, quartis, mínimo e máximo)
print(dados.describe())

# Histogramas das variáveis numéricas
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
sns.histplot(dados['Vendas_Cafe'], kde=True, color='skyblue', ax=axes[0])
axes[0].set_title('Distribuição das Vendas de Café')

sns.histplot(dados['Preco_Cafe'], kde=True, color='salmon', ax=axes[1])
axes[1].set_title('Distribuição dos Preços do Café')

sns.histplot(dados['Preco_Leite'], kde=True, color='lightgreen', ax=axes[2])
axes[2].set_title('Distribuição dos Preços do Leite')

plt.tight_layout()
plt.show()

In [ ]:
# Relação entre Preço do Café e Vendas, destacando dias com e sem Promoção
plt.figure(figsize=(8, 5))
sns.scatterplot(
    data=dados,
    x='Preco_Cafe',
    y='Vendas_Cafe',
    hue='Promocao',
    palette={'Sim': 'red', 'Nao': 'black'},
    s=80
)
plt.title('Impacto do Preço e da Promoção nas Vendas de Café')
plt.xlabel('Preço do Café (R$)')
plt.ylabel('Vendas de Café (unidades)')
plt.show()

In [ ]:
# Quantas vezes as vendas ficaram acima ou abaixo da média histórica?
media_vendas = dados['Vendas_Cafe'].mean()
dados['Status_Media'] = np.where(dados['Vendas_Cafe'] > media_vendas, 'Acima da média', 'Abaixo da média')

print(f'Média histórica de vendas: {media_vendas:.2f}')
print(dados['Status_Media'].value_counts())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.countplot(data=dados, x='Status_Media', palette=['gray', 'orange'], ax=axes[0])
axes[0].set_title('Vendas em Relação à Média')

sns.boxplot(data=dados, x='Promocao', y='Vendas_Cafe', palette=['lightgray', 'salmon'], ax=axes[1])
axes[1].set_title('Vendas com Promoção vs Sem Promoção')

plt.tight_layout()
plt.show()

### 2.2 Regressão Linear Simples (`Vendas_Cafe ~ Preco_Cafe`)

In [ ]:
# Ajuste do modelo linear simples
modelo_simples_cafe = smf.ols('Vendas_Cafe ~ Preco_Cafe', data=dados).fit()
print(modelo_simples_cafe.summary())

# Gráfico com a reta de regressão ajustada
sns.lmplot(
    data=dados,
    x='Preco_Cafe',
    y='Vendas_Cafe',
    hue='Promocao',
    palette={'Sim': 'red', 'Nao': 'black'},
    fit_reg=False,
    height=5,
    aspect=1.4
)
# Sobrepor a reta global do modelo simples
x_vals = np.linspace(dados['Preco_Cafe'].min(), dados['Preco_Cafe'].max(), 100)
y_vals = modelo_simples_cafe.params['Intercept'] + modelo_simples_cafe.params['Preco_Cafe'] * x_vals
plt.plot(x_vals, y_vals, color='blue', linewidth=2, label='Linha de Regressão Simples')
plt.title('Linha de Regressão: Preço vs Vendas')
plt.legend()
plt.show()

### 2.3 Regressão Linear Múltipla e Variáveis Categóricas (Dummies)

Para incluir uma variável categórica (texto) como `Promocao` no modelo matemático, o `statsmodels` utiliza a função **`C(Promocao)`** na fórmula, convertendo automaticamente as categorias em **variáveis dummy** ($0$ e $1$).

In [ ]:
# Ajuste da regressão múltipla com 3 variáveis explicativas
modelo_multiplo_cafe = smf.ols('Vendas_Cafe ~ Preco_Cafe + C(Promocao) + Preco_Leite', data=dados).fit()
print(modelo_multiplo_cafe.summary())

### 2.4 Gráfico 3D e Plano de Regressão (`Preco_Cafe` e `Preco_Leite`)

In [ ]:
# Filtrar casos completos (sem valores ausentes)
dados_plot = dados[['Preco_Cafe', 'Preco_Leite', 'Vendas_Cafe']].dropna()

# Ajustar modelo com duas preditoras numéricas para o plano 3D
modelo_3d_cafe = smf.ols('Vendas_Cafe ~ Preco_Cafe + Preco_Leite', data=dados_plot).fit()

fig = plt.figure(figsize=(9, 6))
ax = fig.add_subplot(111, projection='3d')

ax.scatter(dados_plot['Preco_Cafe'], dados_plot['Preco_Leite'], dados_plot['Vendas_Cafe'], color='darkgreen', s=50)

# Malha para desenhar o plano de regressão
x_surf, y_surf = np.meshgrid(
    np.linspace(dados_plot['Preco_Cafe'].min(), dados_plot['Preco_Cafe'].max(), 20),
    np.linspace(dados_plot['Preco_Leite'].min(), dados_plot['Preco_Leite'].max(), 20)
)
z_surf = (
    modelo_3d_cafe.params['Intercept']
    + modelo_3d_cafe.params['Preco_Cafe'] * x_surf
    + modelo_3d_cafe.params['Preco_Leite'] * y_surf
)
ax.plot_surface(x_surf, y_surf, z_surf, alpha=0.3, color='cyan')

ax.set_xlabel('Preço do Café')
ax.set_ylabel('Preço do Leite')
ax.set_zlabel('Vendas de Café')
ax.set_title('Plano de Regressão Múltipla (3D)')
plt.show()

### 2.5 Visualização de Interações (`Preco_Cafe * C(Promocao)`)

Quando suspeitamos que o efeito do preço do café muda dependendo de haver ou não promoção ativa, incluímos um **termo de interação** (`Preco_Cafe * C(Promocao)`).

In [ ]:
# Modelo com termo de interação entre Preco_Cafe e Promocao
modelo_interacao = smf.ols('Vendas_Cafe ~ Preco_Cafe * C(Promocao) + Preco_Leite', data=dados).fit()
print(modelo_interacao.summary())

# Visualizar as inclinações distintas para 'Sim' e 'Nao'
sns.lmplot(
    data=dados,
    x='Preco_Cafe',
    y='Vendas_Cafe',
    hue='Promocao',
    palette={'Sim': 'red', 'Nao': 'black'},
    height=5,
    aspect=1.4
)
plt.title('Efeito de Interação: Preço do Café x Promoção')
plt.show()

### Questões para Análise — Projeto Cafeteria
1. O preço do café influencia a venda? Qual o sentido e a magnitude desse efeito?
2. Promoções fazem diferença estatisticamente significativa nas vendas?
3. Analise o erro residual e o $R^2$ ajustado: o modelo múltiplo melhora a capacidade explicativa em relação ao simples?
4. Que recomendações práticas você daria à cafeteria para manter ou ampliar as vendas de café?

---
## 3. Projeto Floricultura (Base `iris`)

Uma floricultura especializada em espécies do gênero *Iris* deseja prever o **comprimento da pétala** (`petal_length`) a partir das dimensões da sépala (`sepal_length`, `sepal_width`) e da espécie da flor (`species`).

In [ ]:
# Carregar a base nativa iris
iris = sns.load_dataset('iris')
print(iris.head())
print(iris.info())
print('\nContagem por espécie:')
print(iris['species'].value_counts())

In [ ]:
# Análise exploratória: Boxplot por espécie e Matriz de Dispersão (pairs)
plt.figure(figsize=(7, 4))
sns.boxplot(data=iris, x='species', y='petal_length', palette='pastel')
plt.title('Petal Length por Espécie')
plt.show()

sns.pairplot(iris, hue='species')
plt.show()

In [ ]:
# Regressão Linear Simples: petal_length ~ sepal_length
mod_iris_simples = smf.ols('petal_length ~ sepal_length', data=iris).fit()
print(mod_iris_simples.summary())

plt.figure(figsize=(7, 5))
sns.regplot(data=iris, x='sepal_length', y='petal_length', line_kws={'color': 'blue'})
plt.title('Regressão Simples: Petal Length vs Sepal Length')
plt.show()

In [ ]:
# Regressão Linear Múltipla: petal_length ~ sepal_length + sepal_width
mod_iris_multiplo = smf.ols('petal_length ~ sepal_length + sepal_width', data=iris).fit()
print(mod_iris_multiplo.summary())

# Gráfico 3D com Plano de Regressão
fig = plt.figure(figsize=(9, 6))
ax = fig.add_subplot(111, projection='3d')

cores_iris = {'setosa': 'red', 'versicolor': 'green', 'virginica': 'blue'}
for especie, grupo in iris.groupby('species'):
    ax.scatter(grupo['sepal_length'], grupo['sepal_width'], grupo['petal_length'],
               label=especie, color=cores_iris[especie], s=45)

x_surf, y_surf = np.meshgrid(
    np.linspace(iris['sepal_length'].min(), iris['sepal_length'].max(), 20),
    np.linspace(iris['sepal_width'].min(), iris['sepal_width'].max(), 20)
)
z_surf = (
    mod_iris_multiplo.params['Intercept']
    + mod_iris_multiplo.params['sepal_length'] * x_surf
    + mod_iris_multiplo.params['sepal_width'] * y_surf
)
ax.plot_surface(x_surf, y_surf, z_surf, alpha=0.25, color='gray')

eq_text = (f"Petal.Length = {mod_iris_multiplo.params['Intercept']:.2f} + "
           f"{mod_iris_multiplo.params['sepal_length']:.2f}*Sepal.Length + "
           f"{mod_iris_multiplo.params['sepal_width']:.2f}*Sepal.Width")
ax.set_title(f'Regressão Múltipla 3D\n{eq_text}', fontsize=10)
ax.set_xlabel('Sepal Length')
ax.set_ylabel('Sepal Width')
ax.set_zlabel('Petal Length')
ax.legend()
plt.show()

### 3.1 Comparação de Modelos via ANOVA e Inclusão de Variável Categórica (`species`)

Utilizamos `sm.stats.anova_lm()` para testar se a inclusão de novas variáveis reduz significativamente a Soma dos Quadrados dos Resíduos (RSS).

In [ ]:
# Comparação ANOVA entre o modelo simples e o múltiplo (sem espécie)
print('--- ANOVA: Modelo Simples vs Múltiplo ---')
print(sm.stats.anova_lm(mod_iris_simples, mod_iris_multiplo))

# Inclusão da variável categórica C(species)
mod_iris_species = smf.ols('petal_length ~ sepal_length + C(species)', data=iris).fit()
print('\n--- Resumo do Modelo com Espécie ---')
print(mod_iris_species.summary())

# Visualização do ajuste separado por espécie
sns.lmplot(data=iris, x='sepal_length', y='petal_length', hue='species', height=5, aspect=1.4)
plt.title('Ajuste por Espécie (Controle do Paradoxo de Simpson)')
plt.show()

---
## 4. Atividade Prática: Concessionária (`mtcars`)

Você trabalha em uma concessionária de carros e deseja entender como características técnicas dos veículos influenciam o consumo de combustível (`mpg` — *miles per gallon*).

**Variáveis principais da base `mtcars`:**
* `mpg`: consumo (milhas por galão)
* `hp`: potência do motor (*horsepower*)
* `wt`: peso do carro (em milhares de libras)
* `cyl`: número de cilindros
* `am`: tipo de transmissão (`0` = automática, `1` = manual)

### Roteiro da Atividade:
1. **Parte 0 — Estatística Descritiva:** Explore a base com `.describe()`, `.info()` e `sns.pairplot()`.
2. **Parte 1 — Regressão Linear Simples:** Ajuste um modelo prevendo `mpg` a partir de `hp`.
3. **Parte 2 — Regressão Linear Múltipla:** Adicione o peso do carro (`wt`) como segunda variável explicativa.
4. **Parte 3 — Gráfico 3D:** Plote os pontos e o plano de regressão. Existe algum veículo extremo (*outlier*) que foge ao plano?
5. **Parte 4 — Interpretação:**
   * O que significa o sinal negativo dos coeficientes de `hp` e `wt`?
   * O modelo com duas variáveis é significativamente melhor que o modelo simples? Justifique usando `sm.stats.anova_lm()`.
   * Que outras variáveis da base poderiam ser testadas para melhorar o modelo?

In [ ]:
# Carregar a base mtcars diretamente pelo statsmodels
mtcars = sm.datasets.get_rdataset('mtcars').data
mtcars.head()

In [ ]:
# Parte 0: Estatística descritiva e matriz de dispersão
# Seu código aqui

In [ ]:
# Parte 1: Regressão Linear Simples (mpg ~ hp)
# Seu código aqui

In [ ]:
# Parte 2: Regressão Linear Múltipla (mpg ~ hp + wt) e comparação via anova_lm()
# Seu código aqui

In [ ]:
# Parte 3: Gráfico 3D com plano de regressão
# Seu código aqui